# Logistic Regression: Breast Cancer Classification

This notebook demonstrates binary classification using Logistic Regression on the Breast Cancer Wisconsin dataset.

**Pipeline:**
1. Load and explore the dataset
2. Feature scaling and preprocessing
3. Train Logistic Regression model
4. Evaluate with confusion matrix, ROC curve, and classification report

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, roc_auc_score, ConfusionMatrixDisplay)

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load and Explore the Data

The dataset contains 30 features computed from digitized images of breast mass fine needle aspirates. The target is binary: malignant (0) or benign (1).

In [ ]:
cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target

print(f"Dataset shape: {df.shape}")
print(f"Classes: {cancer.target_names}")
print(f"\nClass distribution:")
print(df['target'].value_counts().rename({0: 'malignant', 1: 'benign'}))
df.head()

In [ ]:
# Visualize distributions of key features by class
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
key_features = ['mean radius', 'mean texture', 'mean perimeter',
                'mean area', 'mean smoothness', 'mean compactness']

for i, feat in enumerate(key_features):
    ax = axes[i // 3][i % 3]
    for label in [0, 1]:
        ax.hist(df[df['target'] == label][feat], bins=30, alpha=0.6,
                label=cancer.target_names[label])
    ax.set_title(feat)
    ax.legend()

plt.tight_layout()
plt.suptitle('Feature Distributions by Class', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Correlation heatmap for mean features
mean_cols = [c for c in df.columns if 'mean' in c] + ['target']
plt.figure(figsize=(10, 8))
sns.heatmap(df[mean_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Correlation Heatmap (Mean Features)')
plt.tight_layout()
plt.show()

## 2. Preprocessing and Model Training

Logistic Regression benefits from feature scaling. We standardize all features to zero mean and unit variance.

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=5000, random_state=42)
model.fit(X_train_scaled, y_train)

print(f"Training accuracy: {model.score(X_train_scaled, y_train):.4f}")
print(f"Test accuracy:     {model.score(X_test_scaled, y_test):.4f}")

## 3. Evaluation: Confusion Matrix and Classification Report

In [ ]:
y_pred = model.predict(X_test_scaled)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
    display_labels=cancer.target_names, cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

## 4. ROC Curve and AUC Score

The ROC curve plots the true positive rate against the false positive rate at various probability thresholds. AUC (Area Under Curve) summarizes the model's discriminative ability.

In [ ]:
y_prob = model.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'r--', lw=1, label='Random classifier')
plt.fill_between(fpr, tpr, alpha=0.1, color='blue')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Top features by coefficient magnitude
coef_df = pd.DataFrame({'feature': cancer.feature_names, 'coefficient': model.coef_[0]})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=True).tail(15)

plt.figure(figsize=(8, 6))
colors = ['green' if c > 0 else 'red' for c in coef_df['coefficient']]
plt.barh(coef_df['feature'], coef_df['coefficient'], color=colors, alpha=0.7)
plt.xlabel('Coefficient Value')
plt.title('Top 15 Feature Coefficients (green=benign, red=malignant)')
plt.tight_layout()
plt.show()